# Aula 02

## Criação do arquivo Excel

In [4]:
# --- Importar as bibliotecas --- #
import openpyxl
from datetime import datetime, date

# --- Criar o Workbook --- #
wb_origem = openpyxl.Workbook()

# --- Definir a aba ativa principal --- #
ws_sp = wb_origem.active
ws_sp.title = 'Faturamento_SP'

# --- Inserir dados com imperfeições --- #
ws_sp.append([])
ws_sp.append([datetime(2026, 3, 10), ' Notebook Pro ', 4500.00, 2])
ws_sp.append([])
ws_sp.append([])
ws_sp.append([datetime(2026, 3, 13), 'monitor gamer', 1899.99, 3])

# --- Adicionar dados à segundo aba/planilha --- #
ws_rj = wb_origem.create_sheet(title='Faturamento_RJ')
ws_rj.append([])
ws_rj.append([datetime(2026, 3, 10), 'Notebook Pro', 4600, 1])
ws_rj.append([])
ws_rj.append([])

# --- Salvar o arquivo Excel --- #
wb_origem.save('dados_vendas.xlsx')
print('Base de dados criada com sucesso!')

Base de dados criada com sucesso!


## Navegação Dinâmica entre Múltiplas Abas

In [5]:
# --- Caminho do arquivo --- #
caminho_arquivo = 'dados_vendas.xlsx'

# --- Carregar o arquivo --- #
wb_leitura = openpyxl.load_workbook(
    filename=caminho_arquivo,  # arquivo a ser carregado
    data_only=True  # mostra somente os dados, se tiver uma fórmula; mostra o resultado, não a fórmula em si
)

# --- Saber quais abas estão ativas no Workbook --- #
abas_disponiveis = wb_leitura.sheetnames
print(f'Abas identificadas de forma dinâmica no arquivo: {abas_disponiveis}')

Abas identificadas de forma dinâmica no arquivo: ['Faturamento_SP', 'Faturamento_RJ']


## Iteração Eficiente via `iter_rows` e Tratamento de Tipos

In [8]:
# --- Iteração por linhas --- #
limite_valor_premium = 2000
faturamento_premium_total = 0

print('--- INICIANDO EXTRAÇÃO DE DADOS POR REGISTRO (ITER_ROWS) ---')
for aba in abas_disponiveis:
    aba_atual = wb_leitura[aba]

    # --- O iter_rows ignora a primeira linha e restringe a busca em 4 colunas --- #
    for linha in aba_atual.iter_rows(min_row=2, max_col=4, values_only=True):
        data_bruta, produto_bruto, preco_bruto, quantidade_bruta = linha

        # --- Validação defensiva contra linhas vazias ou nulas --- #
        if data_bruta is None or produto_bruto is None or preco_bruto is None:
            continue

        # --- Tratamento de strings: eliminação de espaçamentos e padronização textual --- #
        produto_limpo = str(produto_bruto).strip().title()

        # --- Tratamento de floats: assegurar coerência numérica para operações matemáticas --- #
        preco_numerico = float(preco_bruto)

        # --- Tratamento de datetimes: conversão para visualização no padrão brasileiro --- #
        if isinstance(data_bruta, (datetime, date)):
            data_formatada = data_bruta.strftime('%d/%m/%Y')
        else:
            data_formatada = str(data_bruta)

        # --- Extração de inteligência: filtro de transações de alto valor (Premium) --- #
        if preco_numerico >= limite_valor_premium:
            subtotal = preco_numerico * int(quantidade_bruta)
            faturamento_premium_total += subtotal
            print(f'[{aba}] Registro Processado: Data {data_formatada} | '
f'Item: {produto_limpo} | Valor: R$ {preco_numerico:,.2f} | '
f'Qtd: {quantidade_bruta} | Subtotal: R$ {subtotal:,.2f}')

print('\nInteligência consolidada:')
print(f'Faturamento total acumulado de itens premium (>= R$ 2.000,00): R$ {faturamento_premium_total:,.2f}')

--- INICIANDO EXTRAÇÃO DE DADOS POR REGISTRO (ITER_ROWS) --- 
[Faturamento_SP] Registro Processado: Data 10/03/2026 | Item: Notebook Pro | Valor: R$ 4,500.00 | Qtd: 2 | Subtotal: R$ 9,000.00
[Faturamento_RJ] Registro Processado: Data 10/03/2026 | Item: Notebook Pro | Valor: R$ 4,600.00 | Qtd: 1 | Subtotal: R$ 4,600.00

Inteligência consolidada:
Faturamento total acumulado de itens premium (>= R$ 2.000,00): R$ 13,600.00


## Inspeção Vertical e Auditoria via `iter_cols`

In [10]:
# --- Inspeção vertical --- #
print('--- INICIADO EXTRAÇÃO DE COLUNAS COM ITER_COLS ---')
for aba in abas_disponiveis:
    aba_atual = wb_leitura[aba]

    # --- O método iter_cols isola apenas a terceira coluna, pulando o cabeçalho --- #
    for coluna_precos in aba_atual.iter_cols(min_row=2, min_col=3, max_col=3, values_only=True):
        # --- Filtrar elementos nulos para evitar insconsistências analíticas --- #
        lista_precos = [float(p) for p in coluna_precos if p is not None]

        # --- Operações estatísticas --- #
        if lista_precos:
            ticket_medio_filial = sum(lista_precos) / len(lista_precos)
            maior_preco_filial = max(lista_precos)
        else:
            ticket_medio_filial = 0
            maior_preco_filial = 0

        print(f'Auditoria da filial: [{aba}] | Ticket médio de venda: R$ {ticket_medio_filial:,.2f} | '
f'Maior preço identificado: R$ {maior_preco_filial:,.2f}')

--- INICIADO EXTRAÇÃO DE COLUNAS COM ITER_COLS ---
Auditoria da filial: [Faturamento_SP] | Ticket médio de venda: R$ 3,199.99 | Maior preço identificado: R$ 4,500.00
Auditoria da filial: [Faturamento_RJ] | Ticket médio de venda: R$ 4,600.00 | Maior preço identificado: R$ 4,600.00
